<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/Appendix_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 訓練ループに高度なテクニックと追加する
- 学習ウォームアップ
- コサイン減衰
- 勾配クリッピング

In [ ]:
! git clone https://github.com/rasbt/LLMs-from-scratch.git

In [ ]:
import sys
sys.path.append('/content/LLMs-from-scratch/appendix-D/01_main-chapter-code')

In [ ]:
# 訓練モデルを初期化
import torch
from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
model.eval();

In [ ]:
# データローダーの初期化
import os
import requests
import urllib.request

file_path = "./the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


In [ ]:
# text_dataをデータローダーに読み込む
from previous_chapters import create_dataloader_v1

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=False,
    num_workers=0,
)

## D.1 学習率ウォームアップ
- モデルの訓練を安定化させることができる
- 学習率の初期値(`initial_lr`)をかなり低く設定し、そこからユーザー指定の最大値(`peak_lr`)まで徐々に引き上げていく
- より小さな重みで訓練を開始すると、訓練段階で安定性を損なうような大きな更新にモデルが遭遇するリスクを減らすことができる

In [ ]:
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

- ウォームアップのステップ数は、通常は全ステップ数の0.1 ~ 20%の間で設定する

In [ ]:
# ステップ数の計算
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps) # 20%のウォームアップ
print(warmup_steps)

- 最初の27回の訓練ステップで学習率を初期値の0.0001から0.01に引き上げるには、20回のウォームアップステップが必要

In [ ]:
# 訓練ループ
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)

# 20回ウォームアップステップごとにinitial_lrをどれくらい増やすかよって決まる
lr_increment = (peak_lr - initial_lr) / warmup_steps

global_step = -1
track_lrs = []

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1

        if global_step < warmup_steps:
            # まだウォームアップ段階であれば学習率を更新
            lr = initial_lr + global_step * lr_increment
        else:
            lr = peak_lr

        for param_group in optimizer.param_groups:
            # 計算した学習率をオプティマイザに適用
            param_group["lr"] = lr

        track_lrs.append(optimizer.param_groups[0]["lr"])

In [ ]:
# 学習率のウォームアップが正しく機能しているか確認
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.xlabel("Steps")
plt.ylabel("Learning rate")
total_training_steps = len(train_loader) * n_epochs
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout()
plt.show()

- 学習率が低い値から始まり、最大値に達するまでに20ステップにわたって上昇していることがわかる

## D.2 コサイン減衰
- 訓練エポックを通じて学習率を調整し、ウォームアップ後はコサイン曲線に従わせる
- よく使用されるコサイン減衰の一種に、学習率をほぼゼロまで減衰させ、コサイン曲線の半周期分の軌跡を模倣させるものがある
- コサイン減衰における学習率の逓減は、モデルが重みを更新するペースを減少させることを目的としている
- コサイン減衰が特に重要なのは、訓練中に損失の極小値をオーバーシュートするリスクを最小限に抑えるのに役立つため